# TinyChirp CNN build pipeline\n
\n
Trains one CNN on log-mel spectrograms, exports an int8 TFLite model, and writes a Rust audio sample file.

In [1]:
from typing import TYPE_CHECKING
from pathlib import Path
import random
import numpy as np
import tensorflow as tf


if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

tf.random.set_seed(3407)
np.random.seed(3407)
random.seed(3407)
REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR = REPO_ROOT / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT = REPO_ROOT / "dataset"
OUT_TFLITE = MODELS_DIR / "cnn_mel_tf.tflite"
OUT_AUDIO_RS = SRC_DIR / "audio_sample.rs"
SAMPLE_RATE = 16000
FRAME_LENGTH = 1024
FRAME_STEP = 256
FFT_LENGTH = 1024
NUM_MEL_BINS = 80
LOWER_EDGE_HERTZ = 80.0
UPPER_EDGE_HERTZ = 8000.0
TARGET_FRAMES = 184
TARGET_AUDIO_LEN = (TARGET_FRAMES - 1) * FRAME_STEP + FRAME_LENGTH
print("Dataset root:", DATASET_ROOT)
print("Model output:", OUT_TFLITE)
print("Audio sample output:", OUT_AUDIO_RS)


2026-04-01 12:00:53.893458: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-01 12:00:53.899052: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-01 12:00:53.986840: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-01 12:00:54.124708: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-01 12:00:54.278410: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

Dataset root: /home/nathan/Documents/tiny-chirp-microflow/dataset
Model output: /home/nathan/Documents/tiny-chirp-microflow/models/cnn_mel_tf.tflite
Audio sample output: /home/nathan/Documents/tiny-chirp-microflow/src/audio_sample.rs


In [2]:
def fix_audio_length(audio):
    audio = tf.squeeze(audio, axis=-1)  # [batch, time]
    audio = audio[:, :TARGET_AUDIO_LEN]
    current_len = tf.shape(audio)[1]
    pad_len = tf.maximum(0, TARGET_AUDIO_LEN - current_len)
    audio = tf.pad(audio, [[0, 0], [0, pad_len]])
    audio = tf.ensure_shape(audio, [None, TARGET_AUDIO_LEN])
    return audio


def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)


def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)


# Build the exact triangular mel bank used in src/spectrogram.rs.
RUST_FFT_BINS = FFT_LENGTH // 2  # 512
mel_edges = np.zeros(NUM_MEL_BINS + 2, dtype=np.int32)
low_mel = hz_to_mel(LOWER_EDGE_HERTZ)
high_mel = hz_to_mel(UPPER_EDGE_HERTZ)
for i in range(NUM_MEL_BINS + 2):
    frac = i / (NUM_MEL_BINS + 1)
    mel = low_mel + frac * (high_mel - low_mel)
    hz = mel_to_hz(mel)
    bin_idx = int(((FRAME_LENGTH + 1.0) * hz) / SAMPLE_RATE)
    mel_edges[i] = min(bin_idx, RUST_FFT_BINS - 1)

rust_mel_matrix = np.zeros((RUST_FFT_BINS, NUM_MEL_BINS), dtype=np.float32)
for m in range(NUM_MEL_BINS):
    left = mel_edges[m]
    center = mel_edges[m + 1]
    right = mel_edges[m + 2]

    for k in range(left, center):
        rust_mel_matrix[k, m] = (k - left) / max(center - left, 1)
    for k in range(center, right):
        rust_mel_matrix[k, m] = (right - k) / max(right - center, 1)

RUST_MEL_MATRIX = tf.constant(rust_mel_matrix, dtype=tf.float32)


def create_log_mel_spectrogram(audio):
    stfts = tf.signal.stft(
        audio,
        frame_length=FRAME_LENGTH,
        frame_step=FRAME_STEP,
        fft_length=FFT_LENGTH,
        # Match Rust hann_window(i): 0.5 - 0.5*cos(2*pi*i/(N-1))
        window_fn=lambda n, dtype: tf.signal.hann_window(n, periodic=False, dtype=dtype),
    )
    # Match Rust path: keep 512 bins (drop Nyquist bin from 513-bin rfft output).
    spectrograms = tf.abs(stfts)[..., :RUST_FFT_BINS]
    mel_spectrograms = tf.tensordot(spectrograms, RUST_MEL_MATRIX, 1)
    mel_spectrograms.set_shape(spectrograms.shape[:-1].concatenate([NUM_MEL_BINS]))
    return tf.math.log(mel_spectrograms + 1e-6)


def to_features(audio, label):
    audio = fix_audio_length(audio)
    spec = create_log_mel_spectrogram(audio)
    spec = tf.ensure_shape(spec, [None, TARGET_FRAMES, NUM_MEL_BINS])
    spec = tf.expand_dims(spec, axis=-1)
    return spec, label


train_ds_raw = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "training",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=32,
    shuffle=True,
    seed=3407,
)
val_ds_raw = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "validation",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=32,
    shuffle=False,
)
test_ds_raw = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "testing",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=32,
    shuffle=False,
)
label_names = np.array(train_ds_raw.class_names)
num_labels = len(label_names)
print("Classes:", label_names)
train_ds = train_ds_raw.map(to_features, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds_raw.map(to_features, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds_raw.map(to_features, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)



Found 11292 files belonging to 2 classes.


2026-04-01 12:00:56.490556: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-01 12:00:56.493561: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Found 1380 files belonging to 2 classes.


2026-04-01 12:00:56.773654: I tensorflow_io/core/kernels/cpu_check.cc:128] Your CPU supports instructions that this TensorFlow IO binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA


Found 1393 files belonging to 2 classes.
Classes: ['non_target' 'target']


In [3]:
model = keras.Sequential([
    keras.layers.Input(shape=(TARGET_FRAMES, NUM_MEL_BINS, 1)),
    keras.layers.Conv2D(4, (3, 3), activation="relu"),
    keras.layers.AveragePooling2D((2, 2)),
    keras.layers.Conv2D(4, (3, 3), activation="relu"),
    keras.layers.AveragePooling2D((2, 2)),
    keras.layers.Reshape((44 * 18 * 4,)),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(num_labels),
])
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
history = model.fit(train_ds, validation_data=val_ds, epochs=1)
print("Test metrics:", model.evaluate(test_ds, verbose=0))



353/353 ━━━━━━━━━━━━━━━━━━━━ 23s 62ms/step - accuracy: 0.9163 - loss: 0.2157 - val_accuracy: 0.9674 - val_loss: 0.1025
Test metrics: [0.09971363097429276, 0.9655420184135437]


In [4]:
# 1. Gather representative data
val_specs = []
for x_batch, _ in test_ds.unbatch().take(100):
    sample = x_batch.numpy().astype(np.float32)
    sample = np.reshape(sample, (1, TARGET_AUDIO_LEN, 1))
    val_specs.append(sample)

def representative_data_gen():
    for sample in val_specs:
        yield [sample]

# 2. WORKAROUND FOR KERNEL CRASH: Save to disk first
# This prevents Keras 3 / MLIR C++ segfaults during INT8 shape inference
model.export("temp_saved_model") # If on older TF, use model.save("temp_saved_model")
converter = tf.lite.TFLiteConverter.from_saved_model("temp_saved_model")

# 3. Setup Converter
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# NOTE: Removed the experimental flag and the tf.device context manager!

# 4. Convert and Save
try:
    tflite_bytes = converter.convert()
    OUT_TFLITE.write_bytes(tflite_bytes)
    print(f"Success! Wrote {OUT_TFLITE}")
except Exception as e:
    print(f"Conversion failed: {e}")

ValueError: cannot reshape array of size 14720 into shape (1,47872,1)

In [ ]:
raw_sample_ds = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "testing",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=1,
    shuffle=False,
)

clips_by_label = {0: [], 1: []}
for audio_batch, label_batch in raw_sample_ds.unbatch():
    label = int(label_batch.numpy())
    if len(clips_by_label[label]) < 2:
        audio = fix_audio_length(tf.expand_dims(audio_batch, 0))[0].numpy().astype(np.float32)
        clips_by_label[label].append(audio)
    if len(clips_by_label[0]) == 2 and len(clips_by_label[1]) == 2:
        break

# Alternating order: non_target, target, non_target, target
ordered = [
    ("non_target", clips_by_label[0][0]),
    ("target----", clips_by_label[1][0]),
    ("non_target", clips_by_label[0][1]),
    ("target----", clips_by_label[1][1]),
]

rs = []
rs.append("// Generated by building/cnn.ipynb\n")
rs.append(f"pub const SAMPLE_RATE: usize = {SAMPLE_RATE};\n\n")
rs.append("pub struct TestClip {\n")
rs.append("    pub expected_label: &'static str,\n")
rs.append("    pub source_file: &'static str,\n")
rs.append("    pub audio: &'static [f32],\n")
rs.append("}\n\n")

for i, (_label, audio) in enumerate(ordered, 1):
    audio_vals = ", ".join(f"{float(v):.8f}" for v in audio)
    rs.append(f"pub const CLIP_{i}: &[f32] = &[{audio_vals}];\n\n")

rs.append("pub const TEST_CLIPS: [TestClip; 4] = [\n")
for i, (label, _audio) in enumerate(ordered, 1):
    rs.append("    TestClip {\n")
    rs.append(f"        expected_label: \"{label}\",\n")
    rs.append(f"        source_file: \"dataset/testing/{label}/sample_{i}.wav\",\n")
    rs.append(f"        audio: CLIP_{i},\n")
    rs.append("    },\n")
rs.append("];\n")

OUT_AUDIO_RS.write_text("".join(rs), encoding="utf-8")
print("Wrote", OUT_AUDIO_RS, "clips=", len(ordered), "samples_per_clip=", len(ordered[0][1]))

